[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/071.%20The%20Classic%20Vehicle%20Routing%20Problem%20%28VRP%29/P71-Tier-5_executed.ipynb)

---


# P71-Tier-5: Integrated Digital Twin System for VRP

## Purpose and scope

This notebook builds a small, illustrative digital-twin system around a Vehicle Routing Problem (VRP) setting. It represents vehicles, customers, sensors, environmental conditions, predictive models, and scenario tests in code. The example demonstrates how operational observations can be collected, summarized, and used to generate forecasts and recommendations.

A digital twin is a digital representation of a physical system that is updated using information about that system. In a deployed logistics operation, data might come from vehicle GPS devices, traffic services, weather stations, and order-management systems. Here, sensor readings and forecasts are simulated; no physical devices, external data feeds, or trained forecasting models are connected.

The notebook proceeds through the following steps:

1. Define the state and sensor/prediction components.
2. Create a VRP instance and its digital representation.
3. Simulate synchronization, predictions, and recommendations.
4. Run what-if scenarios and display diagnostic metrics.
5. Visualize the system and discuss what the results do and do not establish.

## Modeling assumptions and limitations

- Locations are illustrative coordinates, and the example does not calculate routes or solve a VRP optimization model.
- Sensor readings and predictive outputs are synthetic demonstrations, not measurements or validated forecasts.
- The scenario simulation updates modeled conditions, but it does not move vehicles, serve customers, change travelled distance, or calculate traffic-dependent travel costs.
- The cost metric is a simple distance-based accounting quantity. Since the example does not simulate vehicle movement, its value does not change during these scenarios.
- Performance indicators are diagnostic placeholders. In particular, on-time performance is sampled synthetically rather than derived from delivery schedules.

These assumptions make the notebook a demonstration of digital-twin data flow and scenario handling, not an operational decision-support system.

In [ ]:
from collections import Counter, deque
from copy import deepcopy
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import random
import seaborn as sns

# Fixed seeds make the notebook's synthetic readings reproducible when run top to bottom.
np.random.seed(71)
random.seed(71)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## 1. Define the digital-twin components

The digital twin stores a snapshot of the modeled operation. A snapshot groups vehicle states, customer states, environmental conditions, and system metrics under a timestamp. Keeping these related data together makes it possible to inspect the system as one coherent state.

An IoT sensor is represented by its type, location, update interval, and a bounded buffer of recent readings. The buffer retains the most recent observations without allowing stored history to grow indefinitely. The sensor readings in this example are generated synthetically.

A predictive model represents a forecasting component. Its simple rules are included to demonstrate how a digital twin can pass current state information to a prediction interface; they are not trained statistical models and their accuracy fields are illustrative metadata.

In [ ]:
@dataclass
class DigitalTwinState:
    """A timestamped snapshot of the modeled logistics operation."""

    timestamp: datetime
    vehicle_states: List[Dict]
    customer_states: List[Dict]
    environmental_conditions: Dict
    system_metrics: Dict


@dataclass
class IoTSensor:
    """A simulated sensor with a bounded buffer of recent readings."""

    id: str
    type: str
    location: Tuple[float, float]
    update_frequency: int
    last_update: datetime
    status: str
    data_buffer: deque = field(default_factory=lambda: deque(maxlen=100))

    def generate_reading(self) -> Dict:
        """Return one synthetic reading appropriate to this sensor type."""
        timestamp = datetime.now()

        if self.type == "gps":
            return {
                "timestamp": timestamp,
                "location": self.location,
                "accuracy": float(np.random.uniform(5, 15)),
                "speed": float(np.random.uniform(0, 80)),
            }
        if self.type == "traffic":
            return {
                "timestamp": timestamp,
                "congestion_level": str(
                    np.random.choice(["low", "medium", "high"], p=[0.5, 0.3, 0.2])
                ),
                "average_speed": float(np.random.uniform(20, 60)),
                "flow_rate": float(np.random.uniform(100, 500)),
            }
        if self.type == "weather":
            return {
                "timestamp": timestamp,
                "temperature": float(np.random.uniform(-10, 35)),
                "humidity": float(np.random.uniform(30, 90)),
                "wind_speed": float(np.random.uniform(0, 25)),
                "precipitation": str(
                    np.random.choice(["none", "light", "heavy"], p=[0.7, 0.2, 0.1])
                ),
            }
        if self.type == "vehicle_telematics":
            return {
                "timestamp": timestamp,
                "fuel_level": float(np.random.uniform(10, 100)),
                "engine_temp": float(np.random.uniform(70, 110)),
                "tire_pressure": float(np.random.uniform(28, 35)),
                "mileage": float(np.random.uniform(50000, 200000)),
            }

        raise ValueError(f"Unsupported sensor type: {self.type}")


@dataclass
class PredictiveModel:
    """A lightweight illustrative forecasting component."""

    name: str
    model_type: str
    accuracy: float
    last_trained: datetime
    prediction_horizon: int

    def predict(self, input_data: Dict) -> Dict:
        """Generate a synthetic forecast from the supplied input data."""
        time_ahead = input_data.get("time_ahead", 30)

        if self.model_type == "traffic_forecast":
            congestion_map = {"low": 0.2, "medium": 0.5, "high": 0.8}
            current_level = congestion_map.get(
                input_data.get("current_congestion", "medium"), 0.5
            )
            time_factor = time_ahead / 60
            predicted_level = current_level + 0.1 * np.sin(time_factor * np.pi)
            predicted_level = max(0.1, min(0.9, predicted_level))

            if predicted_level < 0.3:
                predicted_congestion = "low"
            elif predicted_level < 0.7:
                predicted_congestion = "medium"
            else:
                predicted_congestion = "high"

            return {
                "predicted_congestion": predicted_congestion,
                "confidence": self.accuracy / 100,
                "predicted_speed": float(np.random.uniform(20, 60)),
                "prediction_time": datetime.now() + timedelta(minutes=time_ahead),
            }

        if self.model_type == "demand_prediction":
            base_demand = input_data.get("current_demand", 50)
            time_factor = time_ahead / 60
            daily_pattern = 0.2 * np.sin(
                (datetime.now().hour + time_factor) * np.pi / 12
            )
            predicted_demand = base_demand * (
                1 + daily_pattern + 0.1 * np.random.normal()
            )
            predicted_demand = max(10, predicted_demand)

            return {
                "predicted_demand": int(predicted_demand),
                "confidence": self.accuracy / 100,
                "demand_trend": "increasing" if daily_pattern > 0 else "decreasing",
                "prediction_time": datetime.now() + timedelta(minutes=time_ahead),
            }

        raise ValueError(f"Unsupported predictive model type: {self.model_type}")

## 2. Build the VRP digital twin

The class below connects the components into one simulated system. Initialization creates the sensor network, registers the illustrative prediction models, and sets an initial state for each vehicle and customer.

The coordinate pairs and demand values describe the example instance. Vehicle capacity and fleet size are stored as problem parameters, but this notebook does not assign customers to routes or check route feasibility. The data are therefore context for the digital-twin demonstration rather than inputs to a route-optimization procedure.

In [ ]:
class DigitalTwinVRP:
    """A simulated digital-twin data flow for a VRP setting."""

    def __init__(
        self,
        depot_location: Tuple[float, float],
        customer_locations: List[Tuple[float, float]],
        customer_demands: List[int],
        vehicle_capacity: int,
        num_vehicles: int = 3,
    ):
        if len(customer_locations) != len(customer_demands):
            raise ValueError("Each customer location must have one demand value.")
        if vehicle_capacity <= 0 or num_vehicles <= 0:
            raise ValueError("Vehicle capacity and fleet size must be positive.")

        self.depot_location = depot_location
        self.customer_locations = customer_locations
        self.customer_demands = customer_demands
        self.vehicle_capacity = vehicle_capacity
        self.num_vehicles = num_vehicles
        self.n_customers = len(customer_locations)

        self.sensors: List[IoTSensor] = []
        self.models: List[PredictiveModel] = []
        self.current_state: Optional[DigitalTwinState] = None
        self.historical_data = deque(maxlen=1000)

        self.sync_frequency = 5
        self.prediction_horizon = 60
        self.anomaly_threshold = 2.0

        self._initialize_sensors()
        self._initialize_models()
        self._initialize_state()

    def _initialize_sensors(self) -> None:
        """Create simulated vehicle, traffic, and weather sensors."""
        now = datetime.now()
        for i in range(self.num_vehicles):
            self.sensors.append(
                IoTSensor(
                    id=f"gps_vehicle_{i}",
                    type="gps",
                    location=self.depot_location,
                    update_frequency=10,
                    last_update=now,
                    status="active",
                )
            )

        traffic_locations = [(30, 30), (70, 70), (50, 20), (80, 50), (20, 80)]
        for i, location in enumerate(traffic_locations):
            self.sensors.append(
                IoTSensor(
                    id=f"traffic_sensor_{i}",
                    type="traffic",
                    location=location,
                    update_frequency=30,
                    last_update=now,
                    status="active",
                )
            )

        weather_locations = [(25, 25), (75, 75), (50, 50)]
        for i, location in enumerate(weather_locations):
            self.sensors.append(
                IoTSensor(
                    id=f"weather_sensor_{i}",
                    type="weather",
                    location=location,
                    update_frequency=300,
                    last_update=now,
                    status="active",
                )
            )

        for i in range(self.num_vehicles):
            self.sensors.append(
                IoTSensor(
                    id=f"telematics_vehicle_{i}",
                    type="vehicle_telematics",
                    location=self.depot_location,
                    update_frequency=60,
                    last_update=now,
                    status="active",
                )
            )

    def _initialize_models(self) -> None:
        """Register the illustrative traffic and demand forecasting models."""
        now = datetime.now()
        self.models = [
            PredictiveModel(
                name="TrafficForecast_v1",
                model_type="traffic_forecast",
                accuracy=85.0,
                last_trained=now - timedelta(days=1),
                prediction_horizon=60,
            ),
            PredictiveModel(
                name="DemandPredictor_v2",
                model_type="demand_prediction",
                accuracy=78.0,
                last_trained=now - timedelta(hours=6),
                prediction_horizon=120,
            ),
        ]

    def _initialize_state(self) -> None:
        """Create the initial vehicle, customer, and environment state."""
        vehicle_states = [
            {
                "id": f"vehicle_{i}",
                "location": self.depot_location,
                "speed": 0.0,
                "fuel_level": 100.0,
                "status": "idle",
                "current_route": [],
                "distance_traveled": 0.0,
            }
            for i in range(self.num_vehicles)
        ]

        customer_states = []
        for i in range(self.n_customers):
            customer_states.append(
                {
                    "id": f"customer_{i}",
                    "location": self.customer_locations[i],
                    "demand": self.customer_demands[i],
                    "service_status": "pending",
                    "service_time": datetime.now(),
                    "priority": str(
                        np.random.choice(
                            ["low", "medium", "high"], p=[0.6, 0.3, 0.1]
                        )
                    ),
                }
            )

        environmental_conditions = {
            "weather": {
                "temperature": 20.0,
                "humidity": 60.0,
                "wind_speed": 10.0,
                "precipitation": "none",
            },
            "traffic": {"overall_congestion": "medium", "average_speed": 40.0},
            "road_conditions": {"quality": "good", "incidents": []},
        }
        system_metrics = {
            "total_distance": 0.0,
            "total_cost": 0.0,
            "service_completion_rate": 0.0,
            "fuel_efficiency": 0.0,
            "on_time_performance": 0.0,
        }

        self.current_state = DigitalTwinState(
            timestamp=datetime.now(),
            vehicle_states=vehicle_states,
            customer_states=customer_states,
            environmental_conditions=environmental_conditions,
            system_metrics=system_metrics,
        )

    def synchronize_with_physical(self) -> Dict:
        """Collect synthetic sensor readings and update the digital state."""
        sync_results = {
            "timestamp": datetime.now(),
            "sensors_updated": 0,
            "anomalies_detected": [],
            "data_quality_score": 0.0,
        }

        for sensor in self.sensors:
            if sensor.status != "active":
                continue
            try:
                sensor.data_buffer.append(sensor.generate_reading())
                sensor.last_update = datetime.now()
                sync_results["sensors_updated"] += 1
            except Exception as error:
                sync_results["anomalies_detected"].append(
                    f"Sensor {sensor.id}: {error}"
                )

        self._update_vehicle_states()
        self._update_environmental_conditions()

        active_sensors = sum(sensor.status == "active" for sensor in self.sensors)
        sync_results["data_quality_score"] = (
            sync_results["sensors_updated"] / max(active_sensors, 1)
        )
        self.current_state.timestamp = sync_results["timestamp"]
        self.historical_data.append(deepcopy(self.current_state))
        return sync_results

    def _update_vehicle_states(self) -> None:
        """Use the latest GPS and telematics readings for each vehicle."""
        for i, vehicle in enumerate(self.current_state.vehicle_states):
            gps_sensor = next(
                (sensor for sensor in self.sensors if sensor.id == f"gps_vehicle_{i}"),
                None,
            )
            if gps_sensor and gps_sensor.data_buffer:
                latest_gps = gps_sensor.data_buffer[-1]
                vehicle["location"] = latest_gps["location"]
                vehicle["speed"] = latest_gps["speed"]

            telematics_sensor = next(
                (
                    sensor
                    for sensor in self.sensors
                    if sensor.id == f"telematics_vehicle_{i}"
                ),
                None,
            )
            if telematics_sensor and telematics_sensor.data_buffer:
                vehicle["fuel_level"] = telematics_sensor.data_buffer[-1]["fuel_level"]

    def _update_environmental_conditions(self) -> None:
        """Aggregate the latest traffic and weather readings."""
        traffic_sensors = [
            sensor
            for sensor in self.sensors
            if sensor.type == "traffic" and sensor.data_buffer
        ]
        if traffic_sensors:
            latest_traffic = [sensor.data_buffer[-1] for sensor in traffic_sensors]
            self.current_state.environmental_conditions["traffic"][
                "average_speed"
            ] = float(np.mean([reading["average_speed"] for reading in latest_traffic]))
            congestion_levels = [
                reading["congestion_level"] for reading in latest_traffic
            ]
            self.current_state.environmental_conditions["traffic"][
                "overall_congestion"
            ] = Counter(congestion_levels).most_common(1)[0][0]

        weather_sensors = [
            sensor
            for sensor in self.sensors
            if sensor.type == "weather" and sensor.data_buffer
        ]
        if weather_sensors:
            latest_weather = [sensor.data_buffer[-1] for sensor in weather_sensors]
            weather = self.current_state.environmental_conditions["weather"]
            weather["temperature"] = float(
                np.mean([reading["temperature"] for reading in latest_weather])
            )
            weather["humidity"] = float(
                np.mean([reading["humidity"] for reading in latest_weather])
            )
            weather["wind_speed"] = float(
                np.mean([reading["wind_speed"] for reading in latest_weather])
            )
            weather["precipitation"] = Counter(
                reading["precipitation"] for reading in latest_weather
            ).most_common(1)[0][0]

    def generate_predictions(self) -> Dict:
        """Generate forecasts and recommendations from the current state."""
        predictions = {
            "timestamp": datetime.now(),
            "traffic_forecasts": [],
            "demand_forecasts": [],
            "recommendations": [],
        }

        traffic_model = next(
            (model for model in self.models if model.model_type == "traffic_forecast"),
            None,
        )
        if traffic_model:
            congestion = self.current_state.environmental_conditions["traffic"][
                "overall_congestion"
            ]
            for time_ahead in [15, 30, 60]:
                predictions["traffic_forecasts"].append(
                    traffic_model.predict(
                        {"current_congestion": congestion, "time_ahead": time_ahead}
                    )
                )

        demand_model = next(
            (model for model in self.models if model.model_type == "demand_prediction"),
            None,
        )
        if demand_model:
            current_demand = sum(
                customer["demand"]
                for customer in self.current_state.customer_states
                if customer["service_status"] == "pending"
            )
            for time_ahead in [30, 60, 120]:
                predictions["demand_forecasts"].append(
                    demand_model.predict(
                        {"current_demand": current_demand, "time_ahead": time_ahead}
                    )
                )

        predictions["recommendations"] = self._generate_recommendations(predictions)
        return predictions

    def _generate_recommendations(self, predictions: Dict) -> List[str]:
        """Turn forecast conditions and vehicle fuel levels into alerts."""
        recommendations = []
        for forecast in predictions["traffic_forecasts"]:
            if (
                forecast["predicted_congestion"] == "high"
                and forecast["confidence"] > 0.7
            ):
                recommendations.append(
                    "High traffic is forecast at "
                    f"{forecast['prediction_time'].strftime('%H:%M')}; "
                    "review route plans."
                )

        for forecast in predictions["demand_forecasts"]:
            if (
                forecast["demand_trend"] == "increasing"
                and forecast["confidence"] > 0.7
            ):
                recommendations.append(
                    "Increasing demand is forecast; review capacity by "
                    f"{forecast['prediction_time'].strftime('%H:%M')}."
                )

        low_fuel_vehicles = [
            vehicle
            for vehicle in self.current_state.vehicle_states
            if vehicle["fuel_level"] < 30
        ]
        if low_fuel_vehicles:
            recommendations.append(
                f"{len(low_fuel_vehicles)} vehicles have less than 30% fuel."
            )
        return recommendations

    def run_scenario_simulation(
        self, scenario_name: str, scenario_params: Dict
    ) -> Dict:
        """Evaluate a synthetic what-if scenario and restore live state afterward."""
        backup_state = deepcopy(self.current_state)
        backup_sensors = deepcopy(self.sensors)
        backup_history = deepcopy(self.historical_data)

        results = {
            "scenario_name": scenario_name,
            "start_time": datetime.now(),
            "baseline_metrics": self._calculate_metrics(),
            "time_series": [],
            "final_metrics": None,
        }
        duration = scenario_params.get("duration", 30)
        time_step = 5

        try:
            self._apply_scenario(scenario_params)
            for step in range(0, duration, time_step):
                sync_result = self.synchronize_with_physical()
                # Reapply environmental or vehicle conditions after sensor aggregation.
                # Demand is changed once so that repeated steps do not compound the surge.
                if scenario_params.get("type", "normal") != "demand_surge":
                    self._apply_scenario(scenario_params)

                predictions = self.generate_predictions()
                results["time_series"].append(
                    {
                        "time": datetime.now() + timedelta(minutes=step),
                        "metrics": self._calculate_metrics(),
                        "predictions": predictions,
                        "sync_quality": sync_result["data_quality_score"],
                    }
                )

            results["final_metrics"] = self._calculate_metrics()
            results["end_time"] = datetime.now()
        finally:
            # Scenario observations belong to the simulation, not the live twin state.
            self.current_state = backup_state
            self.sensors = backup_sensors
            self.historical_data = backup_history

        return results

    def _apply_scenario(self, scenario_params: Dict) -> None:
        """Apply one of the defined synthetic perturbations to the twin state."""
        scenario_type = scenario_params.get("type", "normal")

        if scenario_type == "high_traffic":
            traffic = self.current_state.environmental_conditions["traffic"]
            traffic["overall_congestion"] = "high"
            traffic["average_speed"] = 20.0
        elif scenario_type == "demand_surge":
            for customer in self.current_state.customer_states:
                if customer["service_status"] == "pending":
                    customer["demand"] = int(customer["demand"] * 1.5)
        elif scenario_type == "vehicle_breakdown":
            count = scenario_params.get("vehicles_affected", 1)
            if count < 0:
                raise ValueError("The number of affected vehicles cannot be negative.")
            for vehicle in self.current_state.vehicle_states[:count]:
                vehicle["status"] = "maintenance"
                vehicle["speed"] = 0.0
        elif scenario_type == "weather_event":
            weather = self.current_state.environmental_conditions["weather"]
            weather["temperature"] = -5.0
            weather["precipitation"] = "heavy"
            self.current_state.environmental_conditions["road_conditions"][
                "quality"
            ] = "poor"
        elif scenario_type != "normal":
            raise ValueError(f"Unsupported scenario type: {scenario_type}")

    def _calculate_metrics(self) -> Dict:
        """Calculate the example's diagnostic metrics from the current state."""
        customers = self.current_state.customer_states
        vehicles = self.current_state.vehicle_states
        completed = sum(
            customer["service_status"] == "completed" for customer in customers
        )
        completion_rate = completed / max(len(customers), 1)

        active_vehicles = [
            vehicle
            for vehicle in vehicles
            if vehicle["status"] in ["en_route", "servicing"]
        ]
        if active_vehicles:
            fuel_efficiency = float(
                np.mean([vehicle["fuel_level"] for vehicle in active_vehicles]) / 100
            )
        else:
            fuel_efficiency = 1.0

        total_distance = sum(vehicle["distance_traveled"] for vehicle in vehicles)
        total_cost = total_distance * 2.5
        on_time_performance = float(
            np.clip(0.85 + 0.1 * np.random.normal(), 0.5, 1.0)
        )

        return {
            "service_completion_rate": completion_rate,
            "fuel_efficiency": fuel_efficiency,
            "total_distance": total_distance,
            "total_cost": total_cost,
            "on_time_performance": on_time_performance,
            "active_vehicles": len(active_vehicles),
            "pending_customers": sum(
                customer["service_status"] == "pending" for customer in customers
            ),
        }

## 3. Create the example system

The example has one depot, ten customers, three vehicles, and a capacity parameter for each vehicle. The total demand is below the combined nominal fleet capacity, but this fact alone does not establish route feasibility: routes, travel costs, and assignment decisions are not computed in this notebook.

The sensor and model counts are derived from the initialized components rather than entered as separate summary values. This makes the displayed system description reflect the actual objects created by the code.

In [ ]:
depot_location = (50, 50)
customer_locations = [
    (20, 30),
    (70, 40),
    (60, 80),
    (30, 70),
    (80, 20),
    (40, 60),
    (75, 75),
    (25, 25),
    (65, 55),
    (45, 45),
]
customer_demands = [10, 15, 12, 8, 20, 18, 14, 16, 11, 9]
vehicle_capacity = 50
num_vehicles = 3

digital_twin = DigitalTwinVRP(
    depot_location=depot_location,
    customer_locations=customer_locations,
    customer_demands=customer_demands,
    vehicle_capacity=vehicle_capacity,
    num_vehicles=num_vehicles,
)

print("Digital Twin VRP System")
print(f"Depot coordinates: {depot_location}")
print(f"Customers: {len(customer_locations)}")
print(f"Total customer demand: {sum(customer_demands)}")
print(f"Vehicle capacity: {vehicle_capacity}")
print(f"Fleet size: {num_vehicles}")
print(f"Simulated sensors: {len(digital_twin.sensors)}")
print(f"Illustrative predictive models: {len(digital_twin.models)}")
print(f"Configured synchronization interval: {digital_twin.sync_frequency} seconds")

sensor_types = Counter(sensor.type for sensor in digital_twin.sensors)
print("\nSensor network:")
for sensor_type, count in sorted(sensor_types.items()):
    print(f"  {sensor_type}: {count}")

print("\nPredictive models:")
for model in digital_twin.models:
    print(
        f"  {model.name}: {model.model_type}; "
        f"illustrative accuracy metadata {model.accuracy:.1f}%; "
        f"horizon {model.prediction_horizon} minutes"
    )

## 4. Synchronize the twin with simulated observations

Synchronization is the process of transferring observations into the digital representation. Each active sensor produces a reading, the reading is appended to its buffer, and the vehicle and environmental states are updated from the latest buffered values. The data-quality score is the fraction of active sensors that produced a reading during this synchronization call.

This score measures successful data collection in the simulation; it does not measure whether the readings are accurate or representative. The vehicle GPS readings also report the sensor's configured location, so this example does not model vehicle motion.

In [ ]:
sync_result = digital_twin.synchronize_with_physical()

print("Synchronization results")
print(f"Timestamp: {sync_result['timestamp'].strftime('%H:%M:%S')}")
print(
    f"Sensors updated: {sync_result['sensors_updated']}/"
    f"{len(digital_twin.sensors)}"
)
print(f"Data-collection score: {sync_result['data_quality_score']:.3f}")
print(f"Sensor errors: {len(sync_result['anomalies_detected'])}")
for anomaly in sync_result["anomalies_detected"]:
    print(f"  - {anomaly}")

state = digital_twin.current_state
print("\nCurrent digital-twin state")
print(
    "Vehicles not marked for maintenance: "
    f"{sum(v['status'] != 'maintenance' for v in state.vehicle_states)}"
)
print(
    "Pending customers: "
    f"{sum(c['service_status'] == 'pending' for c in state.customer_states)}"
)
print(f"Temperature: {state.environmental_conditions['weather']['temperature']:.1f} °C")
print(
    "Overall traffic category: "
    f"{state.environmental_conditions['traffic']['overall_congestion']}"
)
print(
    "Mean traffic-sensor speed: "
    f"{state.environmental_conditions['traffic']['average_speed']:.1f} km/h"
)

print("\nVehicle readings")
for index, vehicle in enumerate(state.vehicle_states, start=1):
    print(
        f"  Vehicle {index}: status={vehicle['status']}, "
        f"fuel={vehicle['fuel_level']:.1f}%, speed={vehicle['speed']:.1f} km/h"
    )

priority_counts = Counter(customer["priority"] for customer in state.customer_states)
print("\nCustomer priority distribution")
for priority, count in sorted(priority_counts.items()):
    print(f"  {priority.capitalize()}: {count}")

## 5. Generate predictive analytics

The prediction step passes the current traffic category and pending customer demand to the corresponding illustrative models. Traffic forecasts use the current congestion category and a simple time-dependent rule. Demand forecasts add a synthetic daily pattern and random variation. The confidence values are copied from model metadata and should not be interpreted as empirically validated probabilities.

Recommendations are rule-based alerts: they are generated when forecast conditions pass fixed thresholds or when a vehicle's recorded fuel level is low. They are prompts for review, not automatically optimized actions.

In [ ]:
predictions = digital_twin.generate_predictions()

print("Traffic forecasts")
for index, forecast in enumerate(predictions["traffic_forecasts"], start=1):
    print(
        f"  {index}. {forecast['prediction_time'].strftime('%H:%M')} — "
        f"congestion: {forecast['predicted_congestion']}; "
        f"confidence metadata: {forecast['confidence']:.2f}; "
        f"synthetic speed: {forecast['predicted_speed']:.1f} km/h"
    )

print("\nDemand forecasts")
for index, forecast in enumerate(predictions["demand_forecasts"], start=1):
    print(
        f"  {index}. {forecast['prediction_time'].strftime('%H:%M')} — "
        f"demand: {forecast['predicted_demand']}; "
        f"trend: {forecast['demand_trend']}; "
        f"confidence metadata: {forecast['confidence']:.2f}"
    )

print("\nRule-based recommendations")
if predictions["recommendations"]:
    for index, recommendation in enumerate(predictions["recommendations"], start=1):
        print(f"  {index}. {recommendation}")
else:
    print("  No recommendation rule was triggered.")

current_metrics = digital_twin._calculate_metrics()
print("\nDiagnostic metrics")
print(f"Service completion: {current_metrics['service_completion_rate']:.1%}")
print(f"Fuel-efficiency proxy: {current_metrics['fuel_efficiency']:.1%}")
print(f"Distance recorded: {current_metrics['total_distance']:.1f} km")
print(f"Distance-based cost: ${current_metrics['total_cost']:.2f}")
print(f"Synthetic on-time proxy: {current_metrics['on_time_performance']:.1%}")
print(f"Active vehicles: {current_metrics['active_vehicles']}")
print(f"Pending customers: {current_metrics['pending_customers']}")

## 6. Run what-if scenario simulations

A scenario simulation temporarily changes part of the digital-twin state, repeats synchronization and prediction steps, records diagnostic metrics, and then restores the state and sensor history that existed before the scenario. This separation lets each scenario start from the same modeled baseline.

The four perturbations represent high traffic, a demand surge, a vehicle breakdown, and adverse weather. Their modeled effects are deliberately limited. For example, traffic and weather conditions do not alter travel time or cost because route movement is not simulated. The comparison is therefore useful for examining state changes and data flow, but it is not evidence of operational cost savings, service impacts, or forecast quality.

In [ ]:
scenario_definitions = [
    (
        "High Traffic",
        {"type": "high_traffic", "duration": 20},
    ),
    (
        "Demand Surge",
        {"type": "demand_surge", "duration": 20},
    ),
    (
        "Vehicle Breakdown",
        {"type": "vehicle_breakdown", "vehicles_affected": 1, "duration": 20},
    ),
    (
        "Adverse Weather",
        {"type": "weather_event", "duration": 20},
    ),
]

scenario_results = {}
for scenario_name, scenario_params in scenario_definitions:
    scenario_results[scenario_name] = digital_twin.run_scenario_simulation(
        scenario_name, scenario_params
    )

    result = scenario_results[scenario_name]
    baseline = result["baseline_metrics"]
    final = result["final_metrics"]
    print(f"\n{scenario_name} scenario")
    print(f"  Simulated duration: {scenario_params['duration']} minutes")
    print(f"  Recorded time steps: {len(result['time_series'])}")
    print(f"  Baseline pending customers: {baseline['pending_customers']}")
    print(f"  Final pending customers: {final['pending_customers']}")
    print(f"  Baseline active vehicles: {baseline['active_vehicles']}")
    print(f"  Final active vehicles: {final['active_vehicles']}")
    print(
        "  Distance-based cost (baseline/final): "
        f"${baseline['total_cost']:.2f} / ${final['total_cost']:.2f}"
    )

print("\nScenario simulations completed. Results are diagnostics of the stated model.")

## 7. Visualize the digital-twin example

The dashboard brings together the modeled locations, sensor-status counts, forecasts, scenario cost diagnostics, performance proxies, and a high-level data-flow diagram. Each panel has labels that explain what is being displayed. The scenario cost panel is expected to be uninformative for these scenarios because the simulation does not move vehicles or change recorded distance.

The visualization summarizes the current in-memory example. It does not represent a live dashboard connected to a physical fleet.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
ax1, ax2, ax3, ax4, ax5, ax6 = axes.flat

# Plot the depot, customers, and current vehicle locations on the example coordinate plane.
ax1.scatter(
    depot_location[0], depot_location[1],
    marker="s", color="black", s=100, label="Depot", zorder=4
)
for index, (x_coord, y_coord) in enumerate(customer_locations):
    customer = digital_twin.current_state.customer_states[index]
    color = "green" if customer["service_status"] == "completed" else "red"
    ax1.scatter(x_coord, y_coord, color=color, s=45, zorder=3)
    ax1.annotate(f"C{index + 1}", (x_coord + 1, y_coord + 1), fontsize=8)
for index, vehicle in enumerate(digital_twin.current_state.vehicle_states, start=1):
    if vehicle["status"] != "maintenance":
        ax1.scatter(
            vehicle["location"][0], vehicle["location"][1],
            marker="^", s=65, label=f"Vehicle {index}", zorder=5
        )
ax1.set_xlabel("X coordinate (illustrative units)")
ax1.set_ylabel("Y coordinate (illustrative units)")
ax1.set_title("Modeled locations")
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# A status count makes the category colors interpretable without relying on color alone.
sensor_status = Counter(sensor.status for sensor in digital_twin.sensors)
status_labels = ["active", "inactive", "maintenance"]
status_values = [sensor_status.get(label, 0) for label in status_labels]
ax2.bar(status_labels, status_values, color=["green", "red", "orange"])
ax2.set_ylabel("Number of sensors")
ax2.set_title("Sensor network status")
ax2.grid(axis="y", alpha=0.3)

traffic_forecasts = predictions["traffic_forecasts"]
forecast_times = [forecast["prediction_time"].strftime("%H:%M") for forecast in traffic_forecasts]
congestion_values = [
    {"low": 0.2, "medium": 0.5, "high": 0.8}[forecast["predicted_congestion"]]
    for forecast in traffic_forecasts
]
ax3.bar(forecast_times, congestion_values, color="darkorange", alpha=0.8)
ax3.set_xlabel("Forecast time")
ax3.set_ylabel("Illustrative congestion index")
ax3.set_ylim(0, 1)
ax3.set_title("Synthetic traffic forecasts")
ax3.grid(axis="y", alpha=0.3)

scenario_names = list(scenario_results)
baseline_costs = [
    scenario_results[name]["baseline_metrics"]["total_cost"]
    for name in scenario_names
]
final_costs = [
    scenario_results[name]["final_metrics"]["total_cost"]
    for name in scenario_names
]
positions = np.arange(len(scenario_names))
bar_width = 0.35
ax4.bar(positions - bar_width / 2, baseline_costs, bar_width, label="Baseline")
ax4.bar(positions + bar_width / 2, final_costs, bar_width, label="Scenario")
ax4.set_xlabel("Scenario")
ax4.set_ylabel("Distance-based cost ($)")
ax4.set_title("Cost diagnostic (distance is not simulated)")
ax4.set_xticks(positions)
ax4.set_xticklabels(scenario_names, rotation=35, ha="right")
ax4.legend()
ax4.grid(axis="y", alpha=0.3)

metric_names = ["Service completion", "Fuel proxy", "On-time proxy"]
metric_values = [
    current_metrics["service_completion_rate"] * 100,
    current_metrics["fuel_efficiency"] * 100,
    current_metrics["on_time_performance"] * 100,
]
metric_bars = ax5.bar(metric_names, metric_values, color=["seagreen", "steelblue", "goldenrod"])
ax5.set_ylabel("Diagnostic value (%)")
ax5.set_ylim(0, 110)
ax5.set_title("Illustrative performance indicators")
ax5.tick_params(axis="x", rotation=20)
ax5.grid(axis="y", alpha=0.3)
for bar, value in zip(metric_bars, metric_values):
    ax5.annotate(
        f"{value:.1f}%",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        fontsize=8,
    )

components = ["Physical\noperation", "Sensors", "Digital\ntwin state", "Forecasts", "Review"]
for index, component in enumerate(components):
    ax6.text(
        0.5,
        index,
        component,
        ha="center",
        va="center",
        bbox={"boxstyle": "round,pad=0.4", "facecolor": "lightblue", "alpha": 0.8},
    )
    if index < len(components) - 1:
        ax6.annotate(
            "",
            xy=(0.5, index + 0.65),
            xytext=(0.5, index + 0.35),
            arrowprops={"arrowstyle": "->", "color": "navy"},
        )
ax6.set_xlim(0, 1)
ax6.set_ylim(-0.5, len(components) - 0.5)
ax6.set_title("Illustrative information flow")
ax6.axis("off")

fig.suptitle("Digital-Twin VRP Demonstration", fontsize=16, fontweight="bold")
fig.tight_layout()
plt.show()

## Interpretation and limitations

The notebook demonstrates a sequence of digital-twin tasks: collect observations, update a state representation, produce forecasts, generate rule-based alerts, and evaluate temporary what-if changes. Its main educational point is the relationship between these components and the data passed between them.

The example does not establish that the forecasts are accurate, that the recommendations improve routes, or that any scenario changes actual operating cost. The model contains no route construction, route optimization, vehicle movement, customer service events, or traffic-dependent travel-time calculations. Its KPI values should therefore be read as software demonstration outputs, not operational performance evidence.

A practical digital twin would require suitable real-time data, validated prediction models, explicitly defined operational events, and a simulation or optimization model whose assumptions match the decisions being evaluated. Those additions would extend the scope of this example.

In [ ]:
print("Digital-twin demonstration complete.")
print(f"Sensors configured: {len(digital_twin.sensors)}")
print(f"Illustrative predictive models: {len(digital_twin.models)}")
print(f"Scenario cases evaluated: {len(scenario_results)}")
print(f"Latest synchronization data-collection score: {sync_result['data_quality_score']:.3f}")
print(
    "Current diagnostic distance-based cost: "
    f"${current_metrics['total_cost']:.2f}"
)
print(
    "Scope reminder: sensor readings, forecasts, and selected performance "
    "indicators are synthetic; no VRP routes are optimized."
)